# Week 03A — Point Operations and Geometric Transforms

**MCTA 4364 Machine Vision** · Session 1 of Week 3 · 80 minutes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W03_preprocessing/W03A_point_ops_geometric_transforms.ipynb) [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W03_preprocessing/W03A_point_ops_geometric_transforms.ipynb)

Pre-processing turns the raw camera image into one that later algorithms can handle reliably. This session covers two families of
operations:
- **Point (intensity) operations** change *what value* each pixel has: contrast, gamma, histogram equalisation, flat-field correction,
  image arithmetic.
- **Geometric transforms** change *where* each pixel goes: translation, rotation, scaling, affine and perspective warps.

### Learning outcomes
By the end of this session you will be able to:
1. Describe point operations as **transfer functions** and implement them efficiently with look-up tables.
2. Derive and implement **histogram equalisation**, and explain when **CLAHE** is better.
3. Correct **uneven illumination** (flat-field) and use **image arithmetic** for noise reduction and template comparison.
4. Name the **hierarchy of 2D transforms** (Euclidean → similarity → affine → projective) with their degrees of freedom.
5. Explain **inverse mapping** and interpolation, and warp images with `warpAffine` and `warpPerspective`.
6. Rectify a document or plane with a **homography**.

### Session plan (80 min)
| Time | Section | Activity |
|---|---|---|
| 0–15 | 1–2 | Point operations, transfer curves, contrast stretching, gamma (interactive) |
| 15–30 | 3 | Histogram equalisation and CLAHE |
| 30–40 | 4–5 | Flat-field correction; image arithmetic |
| 40–55 | 6–7 | Transform hierarchy; forward vs inverse mapping |
| 55–72 | 8 | Homography: document scanner, bird's-eye view |
| 72–80 | 11 | Quiz and exit ticket |
| Home | 9–10 | Exercises with self-checks, challenge |

## 0. Setup

In [ ]:
# Setup: works in Google Colab, Kaggle, Jupyter/VS Code on your laptop, and CI.
import importlib.util
import os
import subprocess
import sys

REPO_URL = "https://github.com/hasanzaki/MCTA-4364-Machine-Vision.git"
if not os.path.exists("resources/scripts/cvhelpers.py"):
    if os.path.exists("../../resources/scripts/cvhelpers.py"):  # opened from its own folder
        os.chdir("../..")
    else:  # Colab / Kaggle: fetch the course helpers
        if not os.path.isdir("MCTA-4364-Machine-Vision"):
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
        os.chdir("MCTA-4364-Machine-Vision")
print("Working directory:", os.getcwd())

# Install only what is missing. PyTorch is deliberately NOT pip-installed here:
# that would replace your laptop's CUDA build with a CPU-only one (see SETUP.md).
needed = {"cv2": "opencv-python", "skimage": "scikit-image", "ipywidgets": "ipywidgets",
          "matplotlib": "matplotlib", "scipy": "scipy", "plotly": "plotly"}
missing = [pkg for mod, pkg in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
sys.path.append("resources/scripts")

In [ ]:
import time

import cv2
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import skimage.data

from cvhelpers import SMOKE, check, interact, load_sample, quiz, show

plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 11})
moon = skimage.data.moon()                     # low-contrast greyscale image (public domain)
cam = skimage.data.camera()
sudoku = load_sample("sudoku.png", gray=True)  # uneven lighting + perspective
show(moon, cam, sudoku, titles=["moon (low contrast)", "camera", "sudoku.png (uneven light, perspective)"], figsize=(17, 5))

## 1. Point operations and transfer functions

A **point operation** computes each output pixel from the *same* input pixel only: $g(x, y) = T\big(f(x, y)\big)$. For 8-bit images,
$T$ is just a table of 256 numbers, a **look-up table (LUT)**, so any point operation, however complicated, costs one table lookup per pixel
(`cv2.LUT`).

| Operation | Transfer function $T(f)$ | Effect |
|---|---|---|
| negative | $255 - f$ | invert (X-ray style) |
| brightness / contrast | $\alpha f + \beta$ | $\beta$ shifts, $\alpha$ stretches |
| gamma | $255\,(f/255)^{\gamma}$ | $\gamma < 1$ brightens shadows, $\gamma > 1$ darkens |
| log | $c\,\log(1 + f)$ | compresses a large dynamic range (e.g. Fourier spectra) |
| contrast stretch | map $[f_\text{lo}, f_\text{hi}] \to [0, 255]$ | uses the full range |
| threshold | $255\cdot[f > t]$ | binarise (Week 4) |

In [ ]:
f = np.arange(256, dtype=np.float32)
lo, hi = np.percentile(moon, [1, 99])
curves = {
    "identity": f, "negative": 255 - f, "contrast x1.8, -60": np.clip(1.8 * f - 60, 0, 255),
    "gamma 0.5 (brighten)": 255 * (f / 255) ** 0.5, "gamma 2.0 (darken)": 255 * (f / 255) ** 2.0,
    "log": 255 * np.log1p(f) / np.log(256), f"stretch [{lo:.0f}, {hi:.0f}]": np.clip((f - lo) / (hi - lo) * 255, 0, 255),
    "threshold 128": (f > 128) * 255.0}
fig, axes = plt.subplots(2, len(curves), figsize=(24, 6.2), gridspec_kw={"height_ratios": [1, 1.25]})
for j, (name, T) in enumerate(curves.items()):
    axes[0, j].plot(f, T, lw=2.5, color="#1a73e8"); axes[0, j].plot(f, f, ":", color="#9aa0a6")
    axes[0, j].set_xlim(0, 255); axes[0, j].set_ylim(0, 255); axes[0, j].set_aspect("equal"); axes[0, j].set_title(name, fontsize=9)
    axes[0, j].set_xticks([0, 128, 255]); axes[0, j].set_yticks([0, 128, 255])
    axes[1, j].imshow(cv2.LUT(moon, T.astype(np.uint8)), cmap="gray", vmin=0, vmax=255); axes[1, j].axis("off")
plt.suptitle("Figure 1.1 - Transfer curves T(f) (top, input on x, output on y) and their effect on the moon image (bottom)", fontweight="bold")
plt.tight_layout(); plt.show()

**LUTs are fast.** Computing a gamma curve with floating-point `pow` for every pixel is much slower than computing 256 values once and looking them up:

In [ ]:
big = cv2.resize(cam, (2000, 2000))
t0 = time.perf_counter(); direct = (255 * (big / 255.0) ** 0.5).astype(np.uint8); t_direct = time.perf_counter() - t0
lut = (255 * (np.arange(256) / 255.0) ** 0.5).astype(np.uint8)
t0 = time.perf_counter(); via_lut = cv2.LUT(big, lut); t_lut = time.perf_counter() - t0
print(f"direct pow(): {t_direct * 1000:.1f} ms | cv2.LUT: {t_lut * 1000:.2f} ms | identical: {np.array_equal(direct, via_lut)}")

## 2. Contrast stretching and gamma

**Contrast stretching** maps the useful intensity range to 0–255. Using the 1st and 99th **percentiles** instead of min/max makes it robust
to a few extreme pixels (dead or hot pixels, specular highlights). **Gamma** is non-linear: cameras apply $\gamma \approx 1/2.2$ so that images
look right on screens. Before *measuring* intensities (e.g. comparing reflectance), you may need to undo it (**linearise**).

In [ ]:
def transfer_demo(alpha=1.0, beta=0, gamma=1.0, image="moon"):
    src = {"moon": moon, "camera": cam, "sudoku": sudoku}[image]
    T = np.clip(255 * (np.clip(alpha * np.arange(256) + beta, 0, 255) / 255) ** gamma, 0, 255).astype(np.uint8)
    out = cv2.LUT(src, T)
    fig, axes = plt.subplots(1, 4, figsize=(19, 4), gridspec_kw={"width_ratios": [1, 1.3, 1, 1.3]})
    axes[0].plot(T, lw=2.5); axes[0].plot([0, 255], [0, 255], ":", color="#9aa0a6"); axes[0].set_aspect("equal")
    axes[0].set_title("transfer curve"); axes[0].set_xlim(0, 255); axes[0].set_ylim(0, 255)
    axes[1].imshow(out, cmap="gray", vmin=0, vmax=255); axes[1].axis("off"); axes[1].set_title("output")
    axes[2].hist(src.ravel(), 64, range=(0, 256), color="#9aa0a6"); axes[2].set_title("input histogram"); axes[2].set_yticks([])
    axes[3].hist(out.ravel(), 64, range=(0, 256), color="#1a73e8"); axes[3].set_title("output histogram"); axes[3].set_yticks([])
    plt.tight_layout(); plt.show()


interact(transfer_demo, alpha=widgets.FloatSlider(value=1.0, min=0.2, max=4.0, step=0.05),
         beta=widgets.IntSlider(value=0, min=-255, max=255, step=5), gamma=widgets.FloatSlider(value=1.0, min=0.2, max=3.0, step=0.05),
         image=["moon", "camera", "sudoku"])

## 3. Histogram equalisation and CLAHE

**Idea:** find the transfer function that makes the output histogram as flat as possible, so every grey level is used equally.
If $p(k)$ is the fraction of pixels with value $k$ and $\text{CDF}(k) = \sum_{j \le k} p(j)$, the mapping

$$ T(k) = \text{round}\big((L - 1)\,\text{CDF}(k)\big), \qquad L = 256 $$

spreads the levels in proportion to how many pixels use them: crowded ranges are stretched, empty ranges compressed. Because the CDF never
decreases, $T$ preserves the order of grey levels.

**Problems with global equalisation:** one mapping for the whole image can over-amplify noise in large flat areas and wash out regions that were
already well exposed. **CLAHE** (Contrast-Limited Adaptive Histogram Equalisation) equalises small **tiles** separately, **clips** each tile's
histogram at a limit (redistributing the excess) so noise is not amplified, and interpolates between tiles to avoid visible seams.

In [ ]:
def cdf(img):
    h = np.bincount(img.ravel(), minlength=256).astype(float)
    return np.cumsum(h) / h.sum()


eq = cv2.equalizeHist(moon)
clahe_img = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(moon)
fig, axes = plt.subplots(2, 3, figsize=(18, 8.5), gridspec_kw={"height_ratios": [1.6, 1]})
for j, (im, name) in enumerate([(moon, "original"), (eq, "global equalisation"), (clahe_img, "CLAHE (clip 2, 8x8 tiles)")]):
    axes[0, j].imshow(im, cmap="gray", vmin=0, vmax=255); axes[0, j].set_title(name); axes[0, j].axis("off")
    axes[1, j].hist(im.ravel(), 128, range=(0, 256), color="#1a73e8"); axes[1, j].set_yticks([])
    ax2 = axes[1, j].twinx(); ax2.plot(cdf(im), color="#d93025", lw=2); ax2.set_ylim(0, 1.05)
plt.suptitle("Figure 3.1 - Equalisation flattens the histogram and straightens the CDF (red)", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# Figure 3.2: where equalisation hurts and CLAHE helps (uneven lighting)
fig, axes = plt.subplots(1, 4, figsize=(21, 5))
panels = [(sudoku, "original"), (cv2.equalizeHist(sudoku), "global equalisation: dark corners stay dark"),
          (cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8)).apply(sudoku), "CLAHE: local contrast everywhere")]
for ax, (im, name) in zip(axes, panels):
    ax.imshow(im, cmap="gray"); ax.set_title(name); ax.axis("off")
ax = axes[3]; ax.imshow(sudoku, cmap="gray", alpha=0.6)
th, tw = sudoku.shape[0] / 8, sudoku.shape[1] / 8
for k in range(1, 8):
    ax.axhline(k * th, color="#d93025", lw=1); ax.axvline(k * tw, color="#d93025", lw=1)
ax.set_title("CLAHE tiles (8 x 8): one histogram each"); ax.axis("off")
plt.suptitle("Figure 3.2 - Global vs adaptive equalisation", fontweight="bold")
plt.tight_layout(); plt.show()

### Interactive: CLAHE parameters
A higher clip limit gives more contrast but amplifies noise; smaller tiles adapt more locally.

In [ ]:
def clahe_demo(clip_limit=2.0, tiles=8, image="sudoku"):
    src = {"moon": moon, "camera": cam, "sudoku": sudoku}[image]
    out = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(tiles, tiles)).apply(src)
    show(src, out, titles=["input", f"CLAHE clip={clip_limit}, {tiles}x{tiles} tiles"], figsize=(13, 5))


interact(clahe_demo, clip_limit=widgets.FloatSlider(value=2.0, min=0.5, max=20, step=0.5),
         tiles=widgets.IntSlider(value=8, min=2, max=32), image=["sudoku", "moon", "camera"])

## 4. Flat-field correction: removing uneven illumination

Vignetting (Week 2) and uneven lamps create a smooth brightness variation $L(x, y)$ across the image: $f = R \cdot L$, where $R$ is the reflectance
we actually care about. If we can estimate $L$, we recover $R \approx f / L$. Two practical estimates:
1. **Calibration:** capture an image of a uniform white target once (the *flat field*).
2. **From the image itself:** heavily smooth it, or apply a morphological **closing/opening** that removes the small dark/bright details (the print)
   while keeping the smooth background (Week 3B covers morphology).

After correction a **single global threshold** works everywhere, which it did not before.

In [ ]:
background = cv2.morphologyEx(sudoku, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (25, 25)))
background = cv2.GaussianBlur(background, (0, 0), 10)
flat = cv2.normalize(sudoku.astype(np.float32) / (background.astype(np.float32) + 1), None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
_, th_raw = cv2.threshold(sudoku, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
_, th_flat = cv2.threshold(flat, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
show(sudoku, background, flat, th_raw, th_flat,
     titles=["input f", "estimated illumination L", "corrected f / L", "global threshold of input", "global threshold after correction"],
     figsize=(24, 5))

## 5. Image arithmetic

**Averaging** $N$ frames of a static scene reduces random noise by a factor $\sqrt{N}$ (the signal adds up $N$ times, independent noise only
$\sqrt{N}$ times). **Subtraction** from a reference ("golden") image highlights *changes*: missing components, extra objects, defects. It only works
if the images are **aligned** and lit identically, which is why fixturing and registration (Weeks 3A and 5) matter.

In [ ]:
rng = np.random.default_rng(0)
clean = cam.astype(np.float32)
frames = [np.clip(clean + rng.normal(0, 30, clean.shape), 0, 255) for _ in range(64)]
Ns = [1, 4, 16, 64]
fig, axes = plt.subplots(1, 5, figsize=(23, 4.5), gridspec_kw={"width_ratios": [1, 1, 1, 1, 1.3]})
noise = []
for ax, n in zip(axes, Ns):
    avg = np.mean(frames[:n], axis=0)
    noise.append(np.std(avg - clean))
    ax.imshow(avg[150:350, 150:350], cmap="gray", vmin=0, vmax=255); ax.set_title(f"average of {n} frame{'s' if n > 1 else ''}"); ax.axis("off")
nn = np.arange(1, 65)
axes[4].loglog(nn, 30 / np.sqrt(nn), "--", color="#9aa0a6", label="theory 30 / sqrt(N)")
axes[4].loglog(Ns, noise, "o", ms=9, color="#d93025", label="measured")
axes[4].set_xlabel("number of frames N"); axes[4].set_ylabel("noise std"); axes[4].legend(); axes[4].grid(alpha=0.3, which="both")
plt.suptitle("Figure 5.1 - Frame averaging reduces noise by sqrt(N)", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# Golden-template inspection of a synthetic PCB
def pcb(defects=False, dx=0):
    board = np.full((240, 360), 40, np.uint8)
    for x in range(40, 330, 60):
        cv2.rectangle(board, (x + dx, 60), (x + 30 + dx, 90), 200, -1)          # pads
        cv2.line(board, (x + 15 + dx, 90), (x + 15 + dx, 170), 150, 4)          # traces
    cv2.circle(board, (180 + dx, 200), 18, 220, -1)
    if defects:
        board[60:90, 160 + dx:190 + dx] = 40                                    # missing pad
        cv2.line(board, (295 + dx, 120), (295 + dx, 128), 40, 6)               # broken trace
        cv2.circle(board, (90 + dx, 200), 5, 230, -1)                           # solder splash
    return np.clip(board + rng.normal(0, 5, board.shape), 0, 255).astype(np.uint8)


golden, test_aligned, test_shifted = pcb(), pcb(True), pcb(True, dx=3)
diff_ok = cv2.absdiff(golden, test_aligned)
diff_bad = cv2.absdiff(golden, test_shifted)
show(golden, test_aligned, cv2.threshold(diff_ok, 60, 255, cv2.THRESH_BINARY)[1], test_shifted,
     cv2.threshold(diff_bad, 60, 255, cv2.THRESH_BINARY)[1],
     titles=["golden reference", "test board (3 defects)", "|difference| > 60: defects found", "test board shifted 3 px",
             "shifted: every edge becomes a 'defect'"], figsize=(24, 4.3))

## 6. Geometric transforms: the 2D hierarchy

In homogeneous coordinates every 2D transform is a 3×3 matrix acting on $(x, y, 1)^\top$, and **composition is matrix multiplication**.

| Transform | Matrix | DoF | Preserves | OpenCV |
|---|---|---|---|---|
| translation | $\begin{bmatrix} 1 & 0 & t_x \\ 0 & 1 & t_y \\ 0&0&1 \end{bmatrix}$ | 2 | everything but position | `warpAffine` |
| Euclidean (rigid) | $\begin{bmatrix} \cos\theta & -\sin\theta & t_x \\ \sin\theta & \cos\theta & t_y \\ 0&0&1\end{bmatrix}$ | 3 | lengths, angles | `estimateAffinePartial2D` (+ scale) |
| similarity | $\begin{bmatrix} s\cos\theta & -s\sin\theta & t_x \\ s\sin\theta & s\cos\theta & t_y \\ 0&0&1\end{bmatrix}$ | 4 | angles, ratios of lengths | `getRotationMatrix2D` |
| affine | $\begin{bmatrix} a & b & t_x \\ c & d & t_y \\ 0&0&1\end{bmatrix}$ | 6 | parallel lines, ratios along a line | `getAffineTransform` (3 points) |
| projective (homography) | $\begin{bmatrix} h_{11} & h_{12} & h_{13} \\ h_{21} & h_{22} & h_{23} \\ h_{31} & h_{32} & 1\end{bmatrix}$ | 8 | straight lines only | `getPerspectiveTransform` (4 points) |

A homography is exactly the mapping between two images of a **plane** (or two images from a camera that only **rotates**), which is why it appears in
calibration (Week 2), document scanning (below) and panorama stitching (Week 5).

In [ ]:
def shape_points():
    square = np.array([[0, 0], [1, 0], [1, 1], [0, 1], [0, 0]], float)
    letter_f = np.array([[0.3, 0.15], [0.3, 0.85], [0.7, 0.85], [0.3, 0.85], [0.3, 0.5], [0.6, 0.5]])
    grid = [np.array([[x, 0], [x, 1]]) for x in np.linspace(0, 1, 5)] + [np.array([[0, y], [1, y]]) for y in np.linspace(0, 1, 5)]
    return square, letter_f, grid


def apply(Hm, pts):
    ph = np.c_[pts, np.ones(len(pts))] @ Hm.T
    return ph[:, :2] / ph[:, 2:3]


th = np.deg2rad(25)
Ts = {"translation (2 DoF)": np.array([[1, 0, 0.4], [0, 1, 0.2], [0, 0, 1]]),
      "Euclidean (3 DoF)": np.array([[np.cos(th), -np.sin(th), 0.4], [np.sin(th), np.cos(th), 0.0], [0, 0, 1]]),
      "similarity (4 DoF)": np.array([[0.6 * np.cos(th), -0.6 * np.sin(th), 0.5], [0.6 * np.sin(th), 0.6 * np.cos(th), 0.1], [0, 0, 1]]),
      "affine (6 DoF)": np.array([[1.0, 0.5, 0.1], [0.2, 0.8, 0.1], [0, 0, 1]]),
      "projective (8 DoF)": np.array([[1.0, 0.2, 0.1], [0.1, 1.0, 0.1], [0.6, 0.4, 1]])}
square, letter_f, grid = shape_points()
fig, axes = plt.subplots(1, len(Ts), figsize=(23, 4.6))
for ax, (name, Hm) in zip(axes, Ts.items()):
    for g in grid:
        ax.plot(*g.T, color="#e8eaed", lw=1); ax.plot(*apply(Hm, g).T, color="#8ab4f8", lw=1)
    ax.plot(*square.T, color="#9aa0a6", lw=1.5, ls="--"); ax.plot(*letter_f.T, color="#9aa0a6", lw=2, ls="--")
    ax.plot(*apply(Hm, square).T, color="#1a73e8", lw=2.5); ax.plot(*apply(Hm, letter_f).T, color="#d93025", lw=3)
    ax.set_aspect("equal"); ax.set_xlim(-0.3, 2.0); ax.set_ylim(-0.3, 1.8); ax.set_title(name); ax.grid(alpha=0.2)
plt.suptitle("Figure 6.1 - The transform hierarchy: each level adds freedom and loses an invariant", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# Figure 6.2: order matters. Rotating then translating differs from translating then rotating.
R = Ts["Euclidean (3 DoF)"].copy(); R[:2, 2] = 0
Tt = Ts["translation (2 DoF)"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
for ax, (Hm, name) in zip(axes, [(Tt @ R, "T @ R: rotate first, then translate"), (R @ Tt, "R @ T: translate first, then rotate")]):
    ax.plot(*letter_f.T, "--", color="#9aa0a6", lw=2); ax.plot(*apply(Hm, letter_f).T, color="#d93025", lw=3)
    ax.plot(0, 0, "k+", ms=12); ax.set_aspect("equal"); ax.set_xlim(-0.4, 1.4); ax.set_ylim(-0.2, 1.6); ax.grid(alpha=0.3)
    ax.set_title(name)
plt.suptitle("Figure 6.2 - Matrix products do not commute: the order of transforms matters", fontweight="bold")
plt.tight_layout(); plt.show()

### Interactive: edit an affine matrix
See how each of the six numbers changes the image. Which parameters create shear? Which keep angles?

In [ ]:
small = cv2.resize(load_sample("building.jpg"), (320, 240))


def affine_demo(a=1.0, b=0.0, tx=0, c=0.0, d=1.0, ty=0):
    M = np.float32([[a, b, tx], [c, d, ty]])
    out = cv2.warpAffine(small, M, (480, 360), borderValue=(235, 235, 235))
    det = a * d - b * c
    show(out, titles=[f"M = [[{a:.2f}, {b:.2f}, {tx}], [{c:.2f}, {d:.2f}, {ty}]]   area scale det = {det:.2f}"], figsize=(8, 6))


interact(affine_demo, a=widgets.FloatSlider(value=1.0, min=-1.5, max=1.5, step=0.05), b=widgets.FloatSlider(value=0.0, min=-1, max=1, step=0.05),
         tx=widgets.IntSlider(value=0, min=-200, max=300), c=widgets.FloatSlider(value=0.0, min=-1, max=1, step=0.05),
         d=widgets.FloatSlider(value=1.0, min=-1.5, max=1.5, step=0.05), ty=widgets.IntSlider(value=0, min=-200, max=300))

## 7. How warping really works: inverse mapping

The obvious approach, **forward mapping** ("for each input pixel, compute where it goes"), leaves **holes** when the image is enlarged or rotated,
because output pixels that no input pixel lands on stay empty. Every real implementation uses **inverse mapping**: *for each output pixel, compute
where it came from* with $M^{-1}$, then **interpolate** the input at that (non-integer) position. This is what `warpAffine`, `warpPerspective`
and `remap` do, and why they take an interpolation flag.

In [ ]:
src = cv2.resize(cam, (160, 160))
M = cv2.getRotationMatrix2D((80, 80), 30, 1.6)
M3 = np.vstack([M, [0, 0, 1]])
forward = np.zeros((160, 160), np.uint8)
ys, xs = np.mgrid[0:160, 0:160]
dst = np.c_[xs.ravel(), ys.ravel(), np.ones(xs.size)] @ M3.T
u, v = np.round(dst[:, 0]).astype(int), np.round(dst[:, 1]).astype(int)
ok = (u >= 0) & (u < 160) & (v >= 0) & (v < 160)
forward[v[ok], u[ok]] = src.ravel()[ok]
inverse_nn = cv2.warpAffine(src, M, (160, 160), flags=cv2.INTER_NEAREST)
inverse_lin = cv2.warpAffine(src, M, (160, 160), flags=cv2.INTER_LINEAR)
show(src, forward, inverse_nn, inverse_lin,
     titles=["input", "FORWARD mapping: holes!", "inverse mapping + nearest", "inverse mapping + bilinear"], figsize=(20, 5))

## 8. Homography in practice

### 8.1 A document scanner in ten lines
To rectify the sudoku we need its four corners. Here we find them automatically, previewing Week 4: adaptive threshold → largest contour →
approximate it by a polygon with 4 vertices. `cv2.getPerspectiveTransform` solves the 8 homography parameters from the 4 correspondences exactly
($4 \times 2 = 8$ equations), and `warpPerspective` applies it.

In [ ]:
def order_corners(pts):
    """Return corners ordered top-left, top-right, bottom-right, bottom-left."""
    pts = pts.reshape(4, 2).astype(np.float32)
    s, d = pts.sum(1), np.diff(pts, axis=1).ravel()
    return np.array([pts[s.argmin()], pts[d.argmin()], pts[s.argmax()], pts[d.argmax()]], np.float32)


binary = cv2.adaptiveThreshold(cv2.GaussianBlur(sudoku, (5, 5), 0), 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 11, 2)
contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
biggest = max(contours, key=cv2.contourArea)
quad = cv2.approxPolyDP(biggest, 0.02 * cv2.arcLength(biggest, True), True)
corners = order_corners(quad)
SIDE = 450
target = np.float32([[0, 0], [SIDE - 1, 0], [SIDE - 1, SIDE - 1], [0, SIDE - 1]])
H = cv2.getPerspectiveTransform(corners, target)
scanned = cv2.warpPerspective(sudoku, H, (SIDE, SIDE))
vis = cv2.cvtColor(sudoku, cv2.COLOR_GRAY2BGR)
cv2.polylines(vis, [corners.astype(int)], True, (0, 0, 255), 3)
for k, p in enumerate(corners):
    cv2.putText(vis, str(k), tuple(p.astype(int) + 5), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 160, 0), 2)
np.set_printoptions(precision=4, suppress=True)
print("Homography H =\n", H)
show(binary, vis, scanned, titles=["adaptive threshold", "largest 4-corner contour", "rectified (warpPerspective)"], figsize=(19, 5.5))

### 8.2 Bird's-eye view of a plane
The same idea turns an oblique camera view of the floor, a conveyor or a table into a metric top-down map. Here we use the chessboard from Week 2B: we map its detected corners to a regular grid.

In [ ]:
chess = load_sample("left12.jpg")
ok, pts = cv2.findChessboardCorners(cv2.cvtColor(chess, cv2.COLOR_BGR2GRAY), (9, 6))
pts = pts.reshape(-1, 2)
S = 40
grid_pts = np.float32([[100 + i * S, 60 + j * S] for j in range(6) for i in range(9)])
Hb, _ = cv2.findHomography(pts, grid_pts)
top = cv2.warpPerspective(chess, Hb, (520, 360))
show(chess, top, titles=["oblique camera view", "bird's-eye view (homography from 54 corners)"], figsize=(15, 5))

### Interactive: drag one corner
Move the top-right target corner and see how a homography differs from an affine warp: straight lines stay straight, but parallel lines no longer stay parallel.

In [ ]:
def homography_demo(dx=0, dy=0):
    tgt = target.copy(); tgt[1] += (dx, dy)
    Hh = cv2.getPerspectiveTransform(corners, tgt)
    out = cv2.warpPerspective(sudoku, Hh, (SIDE + 150, SIDE + 150), borderValue=235)
    show(out, titles=[f"top-right corner moved by ({dx}, {dy})"], figsize=(7, 7))


interact(homography_demo, dx=widgets.IntSlider(value=0, min=-200, max=150, step=5), dy=widgets.IntSlider(value=0, min=-100, max=150, step=5))

## 9. Exercises (with self-checks)
Replace each `raise NotImplementedError`, then run the check below it.

### E1. Histogram equalisation from scratch
Implement $T(k) = \text{round}(255 \cdot \text{CDF}(k))$ with NumPy and apply it as a LUT. Do not call `cv2.equalizeHist`.

In [ ]:
def equalise(gray_img):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e1():
    for im in (moon, sudoku):
        out = equalise(im)
        assert out.dtype == np.uint8 and out.shape == im.shape, "return a uint8 image of the same shape"
        # 1) order preserving: brighter inputs never become darker outputs
        order = np.argsort(im.ravel(), kind="stable")
        assert np.all(np.diff(out.ravel()[order].astype(int)) >= 0), "the mapping must be monotonic (order-preserving)"
        # 2) the output CDF should be close to a straight line
        err = np.abs(cdf(out) - np.arange(256) / 255).max()
        ref = np.abs(cdf(cv2.equalizeHist(im)) - np.arange(256) / 255).max()   # a discrete result is never perfectly flat
        assert err <= ref + 0.03, f"output CDF deviates from uniform by {err:.2f} (OpenCV: {ref:.2f}): is T based on the cumulative histogram?"


check("E1 equalise", _test_e1)

### E2. Rotation about a point
Return the 2×3 matrix that rotates by `angle_deg` (counter-clockwise on screen, like OpenCV) about `center` and scales by `scale`. Build it from translation and rotation matrices; do not call `cv2.getRotationMatrix2D`.

In [ ]:
def rotation_about(center, angle_deg, scale=1.0):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e2():
    for c, a, s in [((80, 80), 30, 1.0), ((10, 200), -45, 0.5), ((320, 240), 90, 2.0)]:
        M = np.asarray(rotation_about(c, a, s), float)
        ref = cv2.getRotationMatrix2D(c, a, s)
        assert M.shape == (2, 3), f"expected a 2x3 matrix, got {M.shape}"
        assert np.allclose(M, ref, atol=1e-9), f"centre {c}, angle {a}, scale {s}: matrix differs from OpenCV. Mind the image y-axis points down."


check("E2 rotation_about", _test_e2)

### E3. Warping by inverse mapping
Implement `warp_affine_nearest(img, M, out_size)` for a greyscale image: for every output pixel, map it back with the **inverse** of `M` and copy the nearest input pixel (0 outside the image). `out_size` is `(width, height)`.

In [ ]:
def warp_affine_nearest(img, M, out_size):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e3():
    img = cv2.resize(cam, (120, 100))
    for M in (cv2.getRotationMatrix2D((60, 50), 25, 1.3), np.float32([[1, 0.3, 5], [0.1, 0.9, -4]])):
        out = np.asarray(warp_affine_nearest(img, M, (150, 130)))
        ref = cv2.warpAffine(img, M, (150, 130), flags=cv2.INTER_NEAREST)
        assert out.shape == ref.shape, f"expected shape {ref.shape}, got {out.shape}"
        agree = np.mean(np.abs(out.astype(int) - ref.astype(int)) <= 2)
        assert agree > 0.97, f"only {agree:.1%} of pixels agree with cv2.warpAffine: did you invert M?"


check("E3 warp_affine_nearest", _test_e3)

### E4. Rotate without cropping (no self-check)
Rotate `small` by 30° about its centre so that **nothing is cut off**: compute the size of the bounding box of the rotated image and add a translation
to the matrix so the result is centred in the enlarged canvas.

In [ ]:
# E4: your code here

## 10. Challenge: a phone-camera document scanner

Photograph a printed page on a desk with your phone (or webcam). Then:
1. Detect the page with the Section 8.1 pipeline (you may need Canny edges, Week 4, if the desk is light).
2. Rectify it to A4 proportions (210 × 297).
3. Apply flat-field correction (Section 4) and CLAHE so the result looks like a scan.
4. Report which step fails most often on your photos and propose a fix.

In [ ]:
# Your challenge code here

## 11. Quick check (auto-marked)

In [ ]:
quiz([
    {
        "q": 'Why are look-up tables (LUTs) used for point operations on 8-bit images?',
        "options": [
            'They improve image quality',
            'Any point operation can be precomputed for all 256 values and applied by table lookup',
            'They are required for colour images',
            'They reduce the image size',
        ],
        "answer": '1de46eea30',
        "explain": '256 evaluations instead of one per pixel.',
    },
    {
        "q": 'Histogram equalisation uses which function of the input as its transfer curve?',
        "options": [
            'The histogram itself',
            'The scaled cumulative distribution function (CDF)',
            'A gamma curve',
            'The mean intensity',
        ],
        "answer": '3f039045b7',
        "explain": 'T(k) = (L - 1) CDF(k).',
    },
    {
        "q": 'Why does CLAHE clip the histogram of each tile?',
        "options": [
            'To save memory',
            'To limit contrast amplification, so noise in flat regions is not amplified',
            'To make the image binary',
            'To remove colour',
        ],
        "answer": '13919ccd17',
        "explain": 'Without clipping, a nearly uniform tile would be stretched enormously.',
    },
    {
        "q": 'Averaging 16 frames of a static scene reduces random noise by what factor?',
        "options": [
            '2',
            '4',
            '8',
            '16',
        ],
        "answer": '9bc6bfb422',
        "explain": 'Noise falls by sqrt(N) = sqrt(16) = 4.',
    },
    {
        "q": 'How many degrees of freedom does a 2D homography have, and how many point pairs determine it?',
        "options": [
            '6 DoF, 3 pairs',
            '8 DoF, 4 pairs',
            '4 DoF, 2 pairs',
            '9 DoF, 5 pairs',
        ],
        "answer": '48093c5a7a',
        "explain": 'A 3x3 matrix up to scale has 8 unknowns; each point pair gives 2 equations.',
    },
    {
        "q": 'Why do warping functions use inverse mapping?',
        "options": [
            'It is easier to write',
            'Every output pixel gets a value, so there are no holes',
            'It avoids interpolation',
            'It preserves colours',
        ],
        "answer": '88ed5ecd3e',
        "explain": 'Forward mapping leaves gaps when the image is enlarged or rotated.',
    },
    {
        "q": 'A golden-template difference shows bright outlines around every feature of a good part. The most likely cause?',
        "options": [
            'The camera is too sharp',
            'The test image is slightly misaligned with the reference',
            'The threshold is too high',
            'The part has many defects',
        ],
        "answer": 'ecd3b54ad7',
        "explain": 'Subtraction needs precise alignment: fixture the part or register the images first.',
    },
], title='Week 03A quiz')

## 12. Exit ticket (reflection)
1. Your inspection images get darker towards the edges. Give a hardware and a software fix.
2. When would you use an affine rather than a projective transform to align two images?
3. Why is it dangerous to apply histogram equalisation *before* measuring intensities (e.g. a colour or grey-level tolerance check)?

## 13. What's new (2025–26)

> - **Learned image enhancement** (low-light enhancement, HDR fusion, denoising) now runs in real time on phone and camera ISPs. For measurement
>   tasks, engineers still prefer transparent, repeatable point operations like those in this notebook.
> - **Learned document rectification** networks unwarp curved and folded pages, which a single homography cannot do.
> - GPU image libraries (e.g. **Kornia** for PyTorch, NVIDIA **VPI**/**CV-CUDA**) implement warps and histogram operations on the GPU so pre-processing
>   no longer bottlenecks deep-learning pipelines.

## 14. References and further exploration

**Textbooks**
- Gonzalez & Woods (2018). *Digital Image Processing*, 4th ed., ch. 3 (intensity transformations, histogram processing) and §2.6 (geometric transforms).
- Szeliski (2022). *Computer Vision: Algorithms and Applications*, 2nd ed., §3.1 (point operators) and §3.6 (geometric transformations). [szeliski.org/Book](https://szeliski.org/Book/)
- Hartley & Zisserman (2004). *Multiple View Geometry*, ch. 2 (projective geometry and the transform hierarchy).
- Burger, W. & Burge, M. J. (2022). *Digital Image Processing: An Algorithmic Introduction*, 3rd ed., ch. 3–4 and 21. Springer.

**Papers**
- Pizer, S. M. et al. (1987). *Adaptive histogram equalization and its variations.* CVGIP 39(3). [doi:10.1016/S0734-189X(87)80186-X](https://doi.org/10.1016/S0734-189X(87)80186-X)
- Zuiderveld, K. (1994). *Contrast Limited Adaptive Histogram Equalization.* In *Graphics Gems IV*, Academic Press (the CLAHE algorithm used by OpenCV).

**Documentation and interactive**
- OpenCV: *Histogram equalization*, *Geometric transformations of images*. [docs.opencv.org](https://docs.opencv.org/4.x/d5/daf/tutorial_py_histogram_equalization.html)
- scikit-image: `exposure` and `transform` modules. [scikit-image.org](https://scikit-image.org/docs/stable/api/skimage.exposure.html)
- Nayar, S. *First Principles of Computer Vision*: "Image Processing I" lectures. [fpcv.cs.columbia.edu](https://fpcv.cs.columbia.edu/)

**Images:** `sudoku.png`, `building.jpg`, `left12.jpg` from the OpenCV sample data; `moon` and `camera` from scikit-image (public domain).
The PCB is generated in code.

## 15. Key takeaways
- Point operations are **transfer curves** $g = T(f)$; implement them as **LUTs**.
- Contrast stretch with **percentiles**; gamma is non-linear; **equalisation** uses the CDF; **CLAHE** adapts locally with a clip limit.
- Correct **uneven illumination** (flat-field) before global thresholds; **average** frames to cut noise by √N; **subtract** a golden template only after alignment.
- 2D transforms form a hierarchy (2, 3, 4, 6, 8 DoF); compose them by **matrix multiplication** (order matters).
- Warps use **inverse mapping + interpolation**; a **homography** rectifies any plane (documents, conveyors, floors).

**Next session (03B):** neighbourhood operations: convolution, smoothing and noise, derivatives and edges, the Fourier view, and morphology.